# Adapting before contact: stage-1a experiment

This notebook runs the first experiment for **CIRA** (Context-Indexed Residual Adapters) end to end in MuJoCo, from the physics up to paper figures and videos.

**The question.** A robot strikes a puck once toward a target near a table edge. The surface (felt, rubber or acrylic) and the load (0.2, 0.6 or 1.0 kg) change during deployment. The robot's frozen world model only knows rubber at 0.6 kg. Methods that adapt from prediction errors, such as AdaJEPA or Sandwich-Residuals with a reset each episode, get no error signal before the strike, so at the moment that decides the task they are exactly the frozen model (Observation 1). CIRA instead keeps a library of residual memories and infers **before contact**, from history and a visual cue, which memory applies.

**What you get.**

| § | Content | Runtime (Colab, `quick`) |
|---|---|---|
| 0 | Setup: code, install, Drive | 2 min |
| 1 | The task: interactive strike, physics check, what the camera sees | 1 min |
| 2 | Deployment streams and cue reliability | < 1 min |
| 3 | Frozen world model: train/load, how visible each context is | 4 min (cached afterwards) |
| 4 | Inside CIRA: one stream step by step, belief heatmap, side-by-side video | 2 min |
| 5 | Main experiment: all methods, parallel and resumable | 12 min |
| 6 | Ablations of CIRA's design choices | 8 min |
| 7 | Playground: your own variants | – |
| 8 | Export for the paper | < 1 min |

**How to run:** `Runtime → Run all`. A GPU runtime (T4 is enough) speeds up frozen-model training and rendering; the experiments themselves run on CPU. With Drive enabled, every finished job is saved immediately and a rerun after a disconnect continues where it stopped.

## 0 · Setup

In [ ]:
# @title Settings { display-mode: "form" }
# smoke: minutes, only checks that everything runs. quick: first real numbers. full: the paper sweep (hours; use Drive).
PROFILE = "quick"  # @param ["smoke", "quick", "full"]
# Keep runs/ (frozen model, results, media) on Google Drive so they survive disconnects.
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/cira_runs"  # @param {type:"string"}
# Retrain the frozen world model even if runs/frozen.pt exists.
RETRAIN_FROZEN = False  # @param {type:"boolean"}
# Private repo: store a GitHub token as the Colab secret GITHUB_TOKEN (key icon on the left), or paste it when asked.
REPO = "github.com/carlo-scr/ethxstanfordresearch.git"  # @param {type:"string"}
# Branch that contains AdaptationContextualInference/code (empty = the repo's default branch).
BRANCH = "research/cira-stage1a"  # @param {type:"string"}

PROFILES = {
    "smoke": dict(episodes=60, streams=1, rhos=[0.8], abl_episodes=40, abl_streams=1),
    "quick": dict(episodes=150, streams=2, rhos=[0.8], abl_episodes=100, abl_streams=1),
    "full": dict(episodes=500, streams=10, rhos=[0.111, 0.5, 0.8, 0.95], abl_episodes=300, abl_streams=5),
}
P = PROFILES[PROFILE]
print(PROFILE, P)

In [ ]:
# Get the code (Colab) or use this checkout (local Jupyter).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = "/content/ethxstanfordresearch"
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    if not token and not os.path.exists(ROOT):
        import getpass
        token = getpass.getpass("GitHub token (leave empty if the repo is public): ")
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

    def git(*args):
        r = subprocess.run(["git", *args], capture_output=True, text=True)
        if r.returncode:
            msg = (r.stderr or r.stdout).replace(token, "***") if token else (r.stderr or r.stdout)
            hint = ("Check that the token has read access to this repo, that BRANCH exists on GitHub, "
                    "and that the code has been pushed.")
            raise RuntimeError(f"git {args[0]} failed:\n{msg.strip()}\n{hint}")
        return r

    branch = ["--branch", BRANCH] if BRANCH else []
    if not os.path.exists(ROOT):
        git("clone", "-q", *branch, url, ROOT)
        git("-C", ROOT, "remote", "set-url", "origin", f"https://{REPO}")  # keep the token out of .git/config
    else:
        git("-C", ROOT, "pull", "-q", url, *([BRANCH] if BRANCH else []))
    if not os.path.isdir(f"{ROOT}/AdaptationContextualInference/code"):
        raise RuntimeError(f"Cloned {BRANCH or 'the default branch'}, but it has no AdaptationContextualInference/code. "
                           "Push the code to that branch (or set BRANCH), delete /content/ethxstanfordresearch, and rerun.")
    os.chdir(f"{ROOT}/AdaptationContextualInference/code")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working directory:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %s (%cr)"], capture_output=True, text=True).stdout)

In [ ]:
# Install. On a CPU-only runtime MuJoCo renders headless through OSMesa, which needs a system library.
import shutil
HAS_GPU = shutil.which("nvidia-smi") is not None
if IN_COLAB:
    if not HAS_GPU:
        subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
sys.path.insert(0, os.getcwd())  # the editable install is only picked up after a restart otherwise
print("GPU:", HAS_GPU, "| CPUs:", os.cpu_count())

In [ ]:
# Optional: keep runs/ on Google Drive.
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    if os.path.isdir("runs") and not os.path.islink("runs"):
        subprocess.run(f"cp -rn runs/. '{DRIVE_DIR}/' && rm -rf runs", shell=True, check=True)
    if not os.path.islink("runs"):
        os.symlink(DRIVE_DIR, "runs")
os.makedirs("runs/media", exist_ok=True)
print("runs ->", os.path.realpath("runs"))

In [ ]:
# Imports (cira first: it picks the headless OpenGL backend before MuJoCo loads) and a test run.
import cira
import mujoco, torch
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dataclasses import replace
from pathlib import Path
from IPython.display import Image as IPImage, Video, display, Markdown

from cira import plots
from cira.config import ExperimentConfig
from cira.env import PuckEnv
from cira.render import SceneRenderer, context_name
from cira.media import strike_image, teaser_strip, obs_grid, comparison_video
from cira.stream import make_stream
from cira.frozen import FrozenModel, collect_training_data, residual_table, rollout_errors
from cira.runner import run_stream
from cira.experiment import Variant, default_variants, run_grid

plots.setup()
cfg = ExperimentConfig()
MEDIA = Path("runs/media")
print("MuJoCo", mujoco.__version__, "| torch", torch.__version__, "| GL backend:", os.environ.get("MUJOCO_GL", "default"))
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])

## 1 · The task

The puck starts on the left; the red disc is the target and the red line is the table edge. The action is one planar impulse **J** (N s). After the strike the robot has no control: the puck slides with Coulomb friction until it stops or falls off. Stopping distance scales as $J^2/(m^2\mu g)$, so mass and friction interact multiplicatively, and the same strike ends very differently on different mats.

Try it: choose a context and an impulse. The ghosts show the trajectory.

In [ ]:
import io
import ipywidgets as W
from PIL import Image as PILImage

_env, _hero = PuckEnv(cfg.env), SceneRenderer(cfg.env, "hero", 960, 540)
n_l = len(cfg.env.loads)
surface = W.Dropdown(options=[(n, i) for i, (n, _) in enumerate(cfg.env.surfaces)], value=1, description="surface")
load = W.Dropdown(options=[(f"{m} kg", i) for i, m in enumerate(cfg.env.loads)], value=1, description="load")
jx = W.FloatSlider(value=1.25, min=0.1, max=3.5, step=0.05, continuous_update=False, description="Jx [N s]")
jy = W.FloatSlider(value=0.0, min=-0.6, max=0.6, step=0.05, continuous_update=False, description="Jy [N s]")
picture, outcome = W.Image(format="png", width=760), W.HTML()

def update(_=None):
    img, _, text = strike_image(_hero, _env, surface.value * n_l + load.value, (jx.value, jy.value))
    buf = io.BytesIO()
    PILImage.fromarray(img).save(buf, "png")
    picture.value, outcome.value = buf.getvalue(), f"<b>{text}</b>"

for w in (surface, load, jx, jy):
    w.observe(update, names="value")
update()
display(W.VBox([W.HBox([surface, load]), W.HBox([jx, jy]), picture, outcome]))

The same strike on the three mats (0.6 kg), which is the core of the teaser figure:

In [ ]:
teaser_strip(cfg, MEDIA)
display(IPImage(str(MEDIA / "teaser_strip.png"), width=1100))

**Physics check.** Friction is an explicit Coulomb force; MuJoCo handles support, the fall over the edge and rendering. (MuJoCo's soft frictional contact made the puck hop under strikes, which buried the friction signal; see the README.) Dots are simulated stopping distances, lines are $v_0^2/(2\mu g)$:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4), layout="constrained")
plots.physics_check(cfg.env, ax)
plt.show()

**What the camera sees.** In stage 1a the methods use the true puck state $[x, y, v_x, v_y]$ and a synthetic cue vector. Stage 1b replaces both with encoder features (DINOv2 / Cosmos) of these plain top-view frames: the mat texture is the surface cue and the puck colour is the load cue.

In [ ]:
obs_grid(cfg, MEDIA)
display(IPImage(str(MEDIA / "obs_grid.png"), width=560))

## 2 · Deployment streams and cues

A stream is a long sequence of episodes. The context persists for a geometrically distributed number of episodes (mean 8), then switches. Before each strike the robot sees a cue: with probability $\rho$ it shows the true context's appearance, otherwise another context's (orange crosses). $\rho = 1/9$ makes the cue useless; $\rho = 0.95$ makes it nearly perfect. Recurring contexts plus an informative cue are exactly the conditions under which the paper's Proposition 1 predicts a first-contact gain.

In [ ]:
demo = make_stream(cfg.env, replace(cfg.stream, n_episodes=150, rho=0.8), seed=0)
fig, ax = plt.subplots(figsize=(9, 3.2), layout="constrained")
plots.stream_timeline(demo, cfg.env, ax)
plt.show()
import time
print(f"{(demo.shown != demo.ctx).mean():.0%} misleading cues, {len(np.unique(demo.ctx))} contexts, "
      f"{(np.diff(demo.ctx) != 0).sum()} switches")

## 3 · The frozen world model

An MLP $f(z, a)$ trained **only on rubber at 0.6 kg**, plus a noise model $\sigma(z, a)$ fitted to its held-out residuals. It stays frozen; every method only adds a residual correction on top. Training data comes from 20k simulated episodes (about 2–3 min), and training runs on the GPU when there is one. The checkpoint is cached in `runs/frozen.pt`.

In [ ]:
from cira.config import FrozenConfig
import time

if RETRAIN_FROZEN or not os.path.exists("runs/frozen.pt"):
    fcfg = FrozenConfig()
    t0 = time.time()
    Z, A, Zn = collect_training_data(cfg.env, fcfg.n_episodes, fcfg.seed)
    print(f"{len(Z)} transitions in {time.time() - t0:.0f}s")
    frozen = FrozenModel(cfg.env, fcfg.hidden, fcfg.sigma_floor).fit(Z, A, Zn, fcfg)
    frozen.save("runs/frozen.pt")
    print(f"trained and saved in {time.time() - t0:.0f}s")
else:
    frozen = FrozenModel.load("runs/frozen.pt")
    print("loaded runs/frozen.pt")
errs = rollout_errors(frozen, cfg.env)
print(f"open-loop rollout on the training condition: median final-position error {100 * np.median(errs):.1f} cm")

**How visible is each context?** Mean $|v|/\sigma$ of the frozen model's velocity residuals, where $v$ is the residual. The training condition sits at about 0.6–0.8, which is pure noise. Light pucks stand out at the strike; contexts that differ from training only in friction (felt or acrylic at 0.6 kg) show up weakly and only while the puck slides. Those are the contexts where the cue matters most.

In [ ]:
table = residual_table(frozen, cfg.env)
fig, ax = plt.subplots(figsize=(4.8, 3.2), layout="constrained")
plots.residual_heatmap(table, cfg.env, ax)
plt.show()

## 4 · Inside CIRA: one stream, step by step

A hand-made stream in which contexts return: heavy rubber → light acrylic → heavy rubber → heavy felt → light acrylic → heavy felt, with nearly reliable cues ($\rho = 0.95$). Watch for three things:

- the **first** episode in a new context misses, because nothing is known yet; a memory is created from what the strike revealed;
- a **returning** context is recognised from the cue and history *before* the strike, so the first strike is already right (recall);
- **Reset-SGD** makes the same strike as the frozen model every time (Observation 1).

In [ ]:
DEMO_SCHEDULE = [5] * 3 + [6] * 3 + [5] * 3 + [2] * 3 + [6] * 3 + [2] * 3
walk = make_stream(cfg.env, replace(cfg.stream, rho=0.95), seed=7, schedule=np.array(DEMO_SCHEDULE))
runs = {m: run_stream(m, walk, frozen, cfg, record=True) for m in ("Reset-SGD", "CIRA")}

tab = pd.DataFrame({
    "context": [context_name(cfg.env, c) for c in walk.ctx],
    "cue ok": walk.shown == walk.ctx,
    **{f"{m} J_x": [r["impulse_x"] for r in rows] for m, (rows, _) in runs.items()},
    **{f"{m} result": ["off table" if r["off_table"] else f"{100 * r['stop_err']:.0f} cm" for r in rows] for m, (rows, _) in runs.items()},
    "CIRA p(true) before contact": [r["p_true"] for r in runs["CIRA"][0]],
    "CIRA memories": [r["n_memories"] for r in runs["CIRA"][0]],
})
display(tab.round(2))

In [ ]:
rows, extras = runs["CIRA"]
fig, ax = plt.subplots(figsize=(9, 2.8), layout="constrained")
plots.belief_heatmap(extras["pre_contact"], extras["memory_labels"], walk, cfg.env, ax)
plt.show()

The same stream as a video (left: Reset-SGD, right: CIRA with its pre-contact belief):

In [ ]:
path = comparison_video(cfg, walk, runs, MEDIA / "reset_vs_cira.mp4", log=None)
Video(str(path), embed=True, width=1000)

## 5 · Main experiment

All methods on the same streams and the same per-episode noise (common random numbers), sharing the planner, the candidate budget and the frozen model:

| method | what it is |
|---|---|
| Frozen | the world model as trained |
| Reset-SGD | residual adapter trained by gradient steps and reset every episode (AdaJEPA / Sandwich-Residuals style) |
| Continual-SGD | the same without resets |
| Cue-RLS | residual regressed on the cue, continually and in closed form (image-to-property style); uses the cue but not the stream's history |
| Mixture | MOLe-style persistent mixture: prediction-error selection, plain CRP, filtered weights, no cue |
| CIRA-NoCue | CIRA without the cue |
| CIRA | the full method (without the factored prior and safety filter, which come later) |
| Oracle | CIRA's memories, selected with the true context |

Jobs run in parallel on all CPUs and each one is saved as soon as it finishes, so rerunning this cell after a disconnect only runs the missing ones.

In [ ]:
t0 = time.time()
df, summary = run_grid(f"main_{PROFILE}", default_variants(), P["rhos"], P["streams"], P["episodes"],
                       "runs/frozen.pt", workers=os.cpu_count())
print(f"done in {(time.time() - t0) / 60:.1f} min")
cols = ["method", "rho", "fc_rel", "strike_err", "success", "success_returns", "off_table", "p_true", "n_memories"]
display(summary[cols].rename(columns={"fc_rel": "first-contact err / Frozen", "success_returns": "success on returns",
                                     "p_true": "p(true) before contact", "n_memories": "memories"}).round(3))

**Reading the table.** `first-contact err / Frozen` is the belief-weighted rollout error of the committed strike relative to the frozen model; Observation 1 predicts exactly 1.00 for Reset-SGD. `success on returns` only counts episodes in a context seen in an earlier block, which is where recall can help. `memories` should approach the number of contexts that occurred (7–9); more means false contexts.

In [ ]:
RHO = min(P["rhos"], key=lambda r: abs(r - 0.8))  # per-rho panels at the rho closest to 0.8
figs = [lambda ax: plots.success_bars(summary, ax, RHO), lambda ax: plots.rolling_error(df, ax, RHO)]
if summary.rho.nunique() > 1:
    figs.insert(0, lambda ax: plots.fc_vs_rho(summary, ax))
fig, axes = plt.subplots(1, len(figs), figsize=(5.4 * len(figs), 3.4), layout="constrained")
for draw, ax in zip(figs, np.atleast_1d(axes)):
    draw(ax)
fig.savefig(f"runs/main_{PROFILE}/overview.png", dpi=200)
plt.show()

## 6 · Ablations

Each row switches off one design choice (see the README for why each was introduced):

- **filtered / hard weights**: update memories with the posterior after the observation (the paper predicts biased, overconfident memories, Remark 1);
- **no episode-conditional scoring**: score hypotheses with the library memories only, so the new-context hypothesis stays at its broad prior;
- **isotropic prior**: drop the physical structure of the weight prior;
- **no merging**: keep duplicate memories;
- **no stickiness**: plain CRP prior over contexts, ignoring that contexts persist.

In [ ]:
ABLATIONS = [
    Variant("CIRA", "CIRA"),
    Variant("filtered weights", "CIRA", {"cira.weights": "filtered"}),
    Variant("hard assignment", "CIRA", {"cira.weights": "hard"}),
    Variant("no episode-conditional", "CIRA", {"cira.episode_conditional": False}),
    Variant("isotropic prior", "CIRA", {"cira.prior": "isotropic"}),
    Variant("no merging", "CIRA", {"cira.merge_z2": 0.0}),
    Variant("no stickiness", "CIRA", {"cira.stay_prob": 0.0}),
]
t0 = time.time()
adf, asum = run_grid(f"ablations_{PROFILE}", ABLATIONS, [0.8], P["abl_streams"], P["abl_episodes"],
                     "runs/frozen.pt", workers=os.cpu_count())
print(f"done in {(time.time() - t0) / 60:.1f} min")
display(asum[["method", "fc_err", "success", "success_returns", "off_table", "p_true", "n_memories"]].round(3))
fig, ax = plt.subplots(figsize=(6, 3.2), layout="constrained")
plots.success_bars(asum, ax, 0.8, metric="success", title="Ablations: strike success (ρ = 0.8)", highlight=("CIRA",))
plt.show()

## 7 · Playground

Any `ExperimentConfig` field can be overridden per variant with `"section.field": value`. The sections are `env`, `stream`, `cira`, `planner` and `sgd` (see `cira/config.py`). Two examples below: a noisier cue, and a longer dwell time. Changing `env` fields that the frozen model depends on (masses, frictions, the training context) requires retraining it (§3).

In [ ]:
RUN_PLAYGROUND = False  # set True to run

if RUN_PLAYGROUND:
    custom = [
        Variant("CIRA", "CIRA"),
        Variant("CIRA, noisy cue", "CIRA", {"stream.cue_noise": 1.0}),
        Variant("CIRA, long dwell", "CIRA", {"stream.mean_dwell": 20.0}),
        Variant("CIRA-NoCue, long dwell", "CIRA-NoCue", {"stream.mean_dwell": 20.0}),
    ]
    cdf, csum = run_grid("playground", custom, [0.8], 1, 100, "runs/frozen.pt", workers=os.cpu_count())
    display(csum[["method", "fc_err", "success", "success_returns", "p_true", "n_memories"]].round(3))

## 8 · Export

`summary.csv` has one row per method and $\rho$, with 95% intervals over streams; point `pgfplots` at it so the figures use the paper's fonts. `episodes.csv` has every episode. The zip below bundles all results and media. With Drive enabled, everything is already in your Drive folder.

In [ ]:
import shutil as _sh, tempfile
bundle = _sh.make_archive(os.path.join(tempfile.gettempdir(), "cira_stage1a_results"), "zip", root_dir="runs", base_dir=".")
print(bundle, f"{os.path.getsize(bundle) / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(bundle)

## Next steps

- **Full sweep**: set `PROFILE = "full"` (500 episodes × 10 streams × 4 cue reliabilities) with Drive on; it resumes across sessions.
- **Composition** (H5): the factored prior over surface × load for unseen combinations.
- **Safety**: latent barrier filter with one ACI margin per memory (H6).
- **Recall vs learning** (H8): the Shapley split of the gain into memory selection and memory updates.
- **Stage 1b**: replace the state with DINOv2 / Cosmos features of the plain top-view frames (§1) and the synthetic cue with encoder features of the mat and puck regions.